# Setup and Checking for Access

In [1]:
!dir

Comprehensive_EDA.ipynb  EDA-test.ipynb


In [2]:
!sudo yum install -y java-11-amazon-corretto-headless

Last metadata expiration check: 0:37:33 ago on Wed Jul  8 00:41:38 2026.
Package java-11-amazon-corretto-headless-1:11.0.31+11-1.amzn2023.x86_64 is already installed.
Dependencies resolved.
Nothing to do.
Complete!


In [3]:
import os
import glob

# Search for the installed Java directory
java_paths = glob.glob('/usr/lib/jvm/java-11*')

if java_paths:
    # Dynamically set the environment variable to the found path
    os.environ["JAVA_HOME"] = java_paths[0]
    print(f"JAVA_HOME successfully set to: {os.environ['JAVA_HOME']}")
else:
    print("Java not found. Did the yum install command work?")

JAVA_HOME successfully set to: /usr/lib/jvm/java-11-amazon-corretto.x86_64


In [23]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, isnan, when, input_file_name
from pyspark.sql.functions import date_format
import pyspark.sql.functions as F



spark = (
    SparkSession.builder
    .appName("DAT204M-EDA-Sagemaker")
    .master("local[*]")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.InstanceProfileCredentialsProvider")
    .config("spark.driver.memory", "10g")
    .getOrCreate()
)

INPUT_PATH = "s3a://dat204m-project-g3/cleaned_data_final/"
OUTPUT_PATH = "s3a://dat204m-project-g3/sampled_eda_data/"


print("Spark ready:", spark.version)

Spark ready: 3.3.0


## Sampling
Considering that our cleaned data is still composed of an estimate of 70GB, we will opt to perform stratified sampling by getting 1% of the data from every month, therefore reducing the amount of data we will be handling for EDA from 70GB to around 700MB. 

In [24]:
# # ==============================================================================
# # DO NOT RUN THIS CELL 
# # This script performs the 70GB stratified sampling and writes to S3. 
# # It was already executed by Imman. 
# # ==============================================================================
# df = spark.read.parquet(INPUT_PATH)

# # Create the stratification key (yyyy-MM)
# df_time = df.withColumn("year_month", date_format("stime", "yyyy-MM"))

# # Dynamically identify all unique months
# # NOTE: This triggers a full read of the 'stime' column across all 180 shards. 
# # It will take a few minutes to compute, but it is necessary for stratification.
# print("Scanning dataset to find all unique months (this will take a moment)...")
# distinct_months_rows = df_time.select("year_month").distinct().collect()

# # Clean the list and remove any potential Null timestamps
# distinct_months = [row["year_month"] for row in distinct_months_rows if row["year_month"] is not None]
# print(f"Found {len(distinct_months)} unique months: {distinct_months}")

# # Set up the sampling fractions (e.g., 0.01 for exactly 1% of every month)
# SAMPLE_FRACTION = 0.01
# fractions = {month: SAMPLE_FRACTION for month in distinct_months}

# # Execute the stratified sample
# print(f"Executing stratified sampling at {SAMPLE_FRACTION*100}% per month...")
# df_sampled = df_time.stat.sampleBy("year_month", fractions, seed=42)

# # Save the sampled dataset back to S3
# print(f"Writing sampled data to: {OUTPUT_PATH}")
# # Using coalesce(4) matches your 4 vCPUs. It prevents Spark from writing 180 tiny files, 
# # instead outputting 4 clean, medium-sized Parquet files for your team to easily read.
# df_sampled.coalesce(4).write.mode("overwrite").parquet(OUTPUT_PATH)

# print("Stratified sampling complete and saved to S3!")

Scanning dataset to find all unique months (this will take a moment)...


Found 6 unique months: ['2023-05', '2023-06', '2023-07', '2023-08', '2023-09', '2023-10']
Executing stratified sampling at 1.0% per month...
Writing sampled data to: s3a://dat204m-project-g3/sampled_eda_data/


Stratified sampling complete and saved to S3!


# Exploratory Data Analysis

We f

In [25]:
EDA_DATA_PATH = "s3a://dat204m-project-g3/sampled_eda_data/"
df = spark.read.parquet(EDA_DATA_PATH)

In [37]:
# Enable clean HTML formatting for Spark DataFrames
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

In [26]:
# ==========================================
# PHASE 1: DIRECTORY VALIDATION (The Proof)
# ==========================================
print("\n--- Validating Directory Contents ---")

# Count total aggregate rows
total_rows = df.count()
print(f"Total Rows in Dataset: {total_rows:,}")


--- Validating Directory Contents ---


[Stage 36:==============================================>         (10 + 2) / 12]

Total Rows in Dataset: 11,430,493


In [27]:
# Extract the unique file names to prove all shards were read
file_distribution = df.withColumn("source_file", input_file_name()) \
    .groupBy("source_file").agg(count("*").alias("row_count"))

total_files_read = file_distribution.count()
print(f"Total Parquet Shards Read: {total_files_read}")

# Display the first 5 shards to verify they match your Athena outputs
print("\nSample of individual files processed:")
file_distribution.show(5, truncate=False)

Total Parquet Shards Read: 4

Sample of individual files processed:


[Stage 45:======================================>                  (8 + 4) / 12]

+-------------------------------------------------------------------------------------------------------------+---------+
|source_file                                                                                                  |row_count|
+-------------------------------------------------------------------------------------------------------------+---------+
|s3a://dat204m-project-g3/sampled_eda_data/part-00000-e4d68cc2-2015-4805-a093-f03af57fd135-c000.snappy.parquet|2864817  |
|s3a://dat204m-project-g3/sampled_eda_data/part-00001-e4d68cc2-2015-4805-a093-f03af57fd135-c000.snappy.parquet|2931512  |
|s3a://dat204m-project-g3/sampled_eda_data/part-00002-e4d68cc2-2015-4805-a093-f03af57fd135-c000.snappy.parquet|2879205  |
|s3a://dat204m-project-g3/sampled_eda_data/part-00003-e4d68cc2-2015-4805-a093-f03af57fd135-c000.snappy.parquet|2754959  |
+-------------------------------------------------------------------------------------------------------------+---------+



In [41]:
# ==========================================
# PHASE 2: COMPREHENSIVE EDA
# ==========================================
print("\n--- Basic Schema ---")
df.printSchema()

print("\n--- Checking for Missing Values ---")
# This calculates the exact number of nulls across the features we kept
missing_val_exprs = [
    count(when(isnan(c) | col(c).isNull(), c)).alias(c)
    for c in df.columns 
    if df.schema[c].dataType.typeName() not in ['timestamp', 'date']
]
df.select(*missing_val_exprs)


--- Basic Schema ---
root
 |-- user_id: long (nullable = true)
 |-- stime: timestamp (nullable = true)
 |-- session_id: string (nullable = true)
 |-- item_id: long (nullable = true)
 |-- event_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- c0_name: string (nullable = true)
 |-- c1_name: string (nullable = true)
 |-- c2_name: string (nullable = true)
 |-- brand_name: string (nullable = true)
 |-- item_condition_name: string (nullable = true)
 |-- year_month: string (nullable = true)


--- Checking for Missing Values ---


user_id,session_id,item_id,event_id,product_id,name,price,c0_name,c1_name,c2_name,brand_name,item_condition_name,year_month
0,0,0,0,0,0,0,0,0,0,0,0,0


In [33]:
print("\n--- Checking for Duplicates ---")

# 1. Group by the specific identifiers and count occurrences
duplicate_groups = df.groupBy("user_id", "stime", "session_id", "item_id").agg(count("*").alias("occurrence_count"))

# 2. Filter to find only the combinations that appear more than once
duplicates = duplicate_groups.filter(col("occurrence_count") > 1)

# 3. Action: Count the total number of duplicate combinations
total_duplicates = duplicates.count()

print(f"Total duplicate combinations found: {total_duplicates:,}")

# 4. Display a sample if duplicates exist
if total_duplicates > 0:
    print("\nSample of duplicated records (sorted by most occurrences):")
    duplicates.orderBy(col("occurrence_count").desc()).show(5, truncate=False)
else:
    print("\nPerfect! No duplicate records found for the given key combination.")


--- Checking for Duplicates ---


[Stage 72:====================================================>   (16 + 1) / 17]

Total duplicate combinations found: 0

Perfect! No duplicate records found for the given key combination.


In [39]:
# Display the first 5 rows
print("\n--- Head of Dataset ---")
df.limit(6)


--- Head of Dataset ---


user_id,stime,session_id,item_id,event_id,product_id,name,price,c0_name,c1_name,c2_name,brand_name,item_condition_name,year_month
54896847,2023-05-20 03:09:46,9cd412bdcea92541e...,231043011,item_view,22105_1101,Puff sleeve tie top,9.0,Women,Tops & blouses,Blouse,SHEIN,Like new,2023-05
54899488,2023-05-04 00:46:25,df7af1950ae474fa6...,87814591,item_view,22635_2990,1986 topps baseba...,80.0,Toys & Collectibles,Sports Trading Cards,Baseball Trading ...,Topps,Like new,2023-05
54902928,2023-05-11 05:15:14,ba782d47de1ba8801...,31205696,item_view,14248_1176,Vintage Miller Be...,35.0,Men,Tops,T-shirts,Miller,Like new,2023-05
56163126,2023-05-15 00:17:19,f665ccf7f3caff832...,224122501,item_view,2693_1339,Bath and Body Wor...,79.0,Beauty,Skin care,Body,Bath & Body Works,New,2023-05
34163376,2023-05-01 17:35:38,ee8638a1b9f0b1f1a...,232158130,item_like,0_464,Coach mini phone ...,80.0,Women,Women's handbags,Crossbody Bags,None,New,2023-05
34164325,2023-05-06 04:44:19,a739ee72e649e0742...,58444656,item_view,12630_2194,LIKE NEW! RARE ♥️...,39.0,Women,Tops & blouses,Tank Tops,lululemon athletica,Like new,2023-05


In [42]:
# Returns a clean list of the unique event names
df.select("name").distinct()

name
Vintage Miller Be...
Michael Kors hand...
Pioneer Woman Cha...
U S Open golf cap
Disney Princess D...
Hell bunny purple...
Lebron James Spac...
Power rangers leg...
Girl clothes size...
Ralph Lauren dress
